# CAMELS-ES: select stations
***

***Author:** Chus Casado Rodríguez*<br>
***Date:** 19-05-2026*<br>

**Introducción:**<br>



**Outputs:**<br>


**To do**:<br>
* [] Filter stations with wrong catchment polygon and fix it.
* [] Probably the timestamps in CERRA are shifted one day (like EMO1).
* [] How to trim the meteo time series at the start? Should I include one extra year as initial condition for the first discharge observation?

In [1]:
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
import geopandas as gpd

from ocab.config import Config

In [2]:
from reservoirs_lshm.utils.timeseries import time_encoding

In [18]:
from typing import Tuple

def time_encoding(
    time: np.ndarray,
    period: int
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Transforms time feature values in an xarray.DataArray to sine and cosine components.

    Parameters:
    -----------
    time: xarray.DataArray
        An xarray.DataArray with time feature values (e.g., month, day of year).
    period: integer
        The period of the time feature (e.g., 12 for months, 7 for days of the week).

    Returns:
    --------
    sin_da, cos_da (tuple of xarray.DataArray):
        Sine and cosine transformations of the time feature values.
    """
    
    # Normalize time feature values to [0, 2π]
    if time.min() == 1:
        norm_da = (time - 1) * 2 * np.pi / period
    elif time.min() == 0:
        norm_da = time * 2 * np.pi / period
    else:
        norm_da = (time - 1) * 2 * np.pi / period
        
    # correct leap years, if necessary
    norm_da = norm_da.where(norm_da <= np.pi * 2, np.pi * 2)
    
    return np.round(np.sin(norm_da), 8), np.round(np.cos(norm_da), 8)

## Configuration

In [4]:
cfg = Config('config_CAMELS_v200.yml')

# point layer
filename = 'stations.geojson'

# paths
path_in = cfg.path_dataset / 'preprocessing' / 'timeseries'

# decimals in output timeseries
rounding = {
    'discharge_cms': 3,
    'discharge_mm': 1,
    'temp_degC': 1,
    'precip_mm': 1,
    'pet_mm': 1,
}

variables = {
    'ta_mean': 'temp_degC', 
    'pr_mean': 'precip_mm', 
    'e0_mean': 'pet_mm',
}

In [ ]:
# metadata to be included in the NetCDF files
metadata = {
    'variables': {
        'discharge_cms': {
            'long_name': 'discharge',
            'units': 'm³/s',
        },
        'discharge_mm': {
            'long_name': 'specific discharge',
            'units': 'mm/d',
        },
        'temp_degC': {
            'long_name': 'mean temperature',
            'units': '°C',
        },
        'pet_mm': {
            'long_name': 'potential evapotranspiration',
            'units': 'mm/d',
        },
        'precip_mm': {
            'long_name': 'precipitation',
            'units': 'mm',
        },
    },
    'Timezone': 'Europe/Madrid',
    'Sources':
        'Discharge: "Anuario de Aforos" curated by CEDEX (Spanish Ministry of Environment) and the River Basin Authorities\n' \
        'ROCIO-IBEB: meteorological observations created by AEMet (Spanish Meteorological Agency)\n' \
        'CERRA-Land: meteorological reanalysis created by C3S (Copernicus Climate Change Service)\n' \
        'EMO1: meteorological observations created by CEMS (Copernicus Emergency Management Service)'
}

## Data

### Stations

In [5]:
# load points
points = gpd.read_file(cfg.path_gis / filename).set_index('id')

print(f'no. points: {len(points)}')

no. points: 1118


### Selection

I load the answers to the questionnaire in the website.

In [6]:
answers = pd.read_excel(cfg.path_dataset / 'selection' / f'{cfg.name} (respuestas).xlsx')

# rename columns
rename_cols = {
    'Marca temporal': 'timestamp', 
    'Station ID': 'ID', 
    'Hydrological regime': 'regime',
    'Start date (1st period)': 'start_1', 
    'End date (1st period) ': 'end_1',
    'Raise any other issue in the station attributes or time series. ': 'comments',
    # 'email', 
    'Is the catchment polygon correct?': 'catchment',
    'Is there a second period of high-quality data?': 'second_period',
    'Start date (2nd period)': 'start_2', 
    'End date (2nd period)': 'end_2',
}
answers.rename(columns=rename_cols, inplace=True)

# keep only selected stations
answers = answers[answers['ID'].isin(points.index)]
print(f'No. answers:\t\t{len(answers)}')
print(f'No. unique stations:\t{len(answers["ID"].unique())}')

# select stations with natural or semi-natural regimes
# if multiple answers, I take the majority vote
IDs = []
for ID in answers['ID'].unique():
    subset = answers[answers['ID'] == ID]
    if len(subset) > 1:
        if subset['regime'].value_counts().index[0] in ['Natural', 'Semi-natural']:
            IDs.append(ID)
    else:
        if subset['regime'].item() in ['Natural', 'Semi-natural']:
            IDs.append(ID)
mask_id = answers['ID'].isin(IDs)
mask_regime = answers['regime'].isin(['Natural', 'Semi-natural'])
answers = answers[mask_id & mask_regime]

print('\nSelect (semi)natural stations:')
print(f'No. answers:\t\t{len(answers)}')
print(f'No. unique stations:\t{len(answers["ID"].unique())}')

# if duplicate stations, keep only the last answer
answers = answers.sort_values('timestamp').drop_duplicates('ID', keep='last')
answers.set_index('ID', inplace=True, drop=True)

No. answers:		330
No. unique stations:	311

Select (semi)natural stations:
No. answers:		239
No. unique stations:	225


In [7]:
def combine_periods(row):
    if row['second_period'] == 'Yes':
        starts = [row['start_1'], row['start_2']]
        ends = [row['end_1'], row['end_2']]
    else:
        starts = [row['start_1']]
        ends = [row['end_1']]
    return pd.Series([starts, ends], index=['starts', 'ends'])

# Apply across the rows (axis=1) to create both columns at once
answers[['starts', 'ends']] = answers.apply(combine_periods, axis=1)

In [8]:
# drop some columns
answers.drop(columns=['timestamp', 'start_1', 'end_1', 'start_2', 'end_2'], inplace=True)

In [9]:
answers['catchment'].value_counts()

catchment
Yes    222
No       3
Name: count, dtype: int64

## Time series

In [11]:
path_csv = cfg.path_timeseries / 'csv' / cfg.prefix
path_nc = cfg.path_timeseries / 'netcdf' / cfg.prefix
for path in [path_csv, path_nc]:
    path.mkdir(parents=True, exist_ok=True)

In [84]:
# process timeseries for each station
for ID in tqdm(answers.index, desc='points'):
    
    # DISCHARGE TIME SERIES
    # .....................
    try:
        dis = pd.read_parquet(path_in / 'discharge' / f'{ID}.parquet')
        dis.columns = ['discharge_cms']
        # compute specific discharge (mm/day)
        dis['discharge_mm'] = dis['discharge_cms'] / points.loc[ID, 'catch_skm'] * 86400 / 1000
        # round values
        dis = dis[dis.columns.intersection(rounding)].round(rounding)
    except Exception as e:
        logger.error(f'Loading discharge timeseries for station {ID:04d}: {e}')
        continue
    
    # METEOROLOGICAL TIME SERIES
    # ..........................
    meteo = pd.DataFrame()
    for dataset in ['ROCIO-IBEB', 'CERRA', 'EMO1']:
        try:
            # read timeseries
            meteo_ts = pd.read_parquet(path_in / 'meteo' / dataset / f'{ID}.parquet').loc[ID]
            # rename variables
            meteo_ts.rename(columns=variables, inplace=True, errors='ignore')
            # round_values
            meteo_ts = meteo_ts[meteo_ts.columns.intersection(rounding)].round(rounding)
            # add dataset suffix
            suffix = dataset.split('-')[0].lower()
            meteo_ts.columns = [f'{col}_{suffix}' for col in meteo_ts.columns]
            # correct dates
            if dataset == 'EMO1':
                meteo_ts.index = meteo_ts.index.date - pd.Timedelta(days=1)
            meteo_ts.index.name = 'date'
            meteo_ts.index = pd.to_datetime(meteo_ts.index)
            # concatenate
            meteo = pd.concat([meteo, meteo_ts], axis=1)
        except Exception as e:
            logger.error(f'Loading {dataset} meteorological timeseries for station {ID}: {e}')
            continue

    # merge timeseries
    start = min(answers.loc[ID, 'starts'])
    end = max(answers.loc[ID, 'ends'])
    ts = pd.concat([dis.loc[start:end], meteo.loc[start:end]], axis=1, )

    # TEMPORAL ENCODERS
    # .................
    ts['year'] = ts.index.year
    ts['month'] = ts.index.month
    ts['month_sin'], ts['month_cos'] = time_encoding(ts['month'], period=12)
    ts['weekofyear'] = ts.index.isocalendar().week.astype('uint32')
    ts['woy_sin'], ts['woy_cos'] = time_encoding(ts['weekofyear'], period=52)
    ts['dayofyear'] = ts.index.dayofyear
    ts['doy_sin'], ts['doy_cos'] = time_encoding(ts['dayofyear'], period=365)
    ts['dayofweek'] = ts.index.isocalendar().day.astype('uint32')
    ts['dow_sin'], ts['dow_cos'] = time_encoding(ts['dayofweek'], period=7)

    # EXPORT
    # ......
    
    # export CSV file
    ts.to_csv(path_csv / f'{cfg.prefix}_{ID}.csv', index=True)

    # export NetCDF file
    ds = ts.to_xarray()
    ds.attrs['Timezone'] = metadata['Timezone']
    ds.attrs['Sources'] = metadata['Sources']
    for var in ds.data_vars:
        var_short = '_'.join(var.split('_')[:2])  # remove dataset suffix
        if var_short in metadata['variables']:
            ds[var].attrs['long_name'] = metadata['variables'][var_short]['long_name']
            ds[var].attrs['units'] = metadata['variables'][var_short]['units']
    ts.to_xarray().to_netcdf(path_nc / f'{cfg.prefix}_{ID}.nc')

points:   0%|          | 0/225 [00:00<?, ?it/s]